In [ ]:
# ---------------------------------------------------------------------------
# Path setup -- fixes imports and data paths when notebook runs from subdir
# ---------------------------------------------------------------------------
from pathlib import Path
import sys, os

REPO_ROOT = Path().resolve().parent.parent  # notebooks/{subdir}/ -> root
os.chdir(REPO_ROOT)                          # all relative paths (cache/, data/, results/, splits/) resolve from root
sys.path.insert(0, str(REPO_ROOT / 'src'))   # find rise, crise, synth_gen modules

# Deepfake Forensics Analysis (Goal 7)

Uses CRISE saliency maps to diagnose *why* synthetic probes succeed or fail at fooling ArcFace.

**Pipeline:**
1. Match synthetic probe saliency maps to real probe saliency maps (same identity)
2. Compute saliency divergence: cosine similarity + L1
3. Establish similarity threshold empirically from the distribution
4. Classify every synthetic probe into case A / B / C / D
5. Per-region importance analysis (5 facial zones on 112×112 aligned chips)
6. Cross-method comparison table + 8+ figures

**Case taxonomy:**

| Case | Fooled ArcFace? | Saliency similar? | Interpretation |
|---|---|---|---|
| A | Yes | Yes | Fooled for right reasons — genuine identity features replicated |
| B | Yes | No  | **Fooled for wrong reasons** — exploiting artifacts / non-identity regions |
| C | No  | Yes | Correct features but insufficient identity transfer |
| D | No  | No  | Complete failure |

**Inputs required (must exist before running):**
- `data/synthetic_probes/metadata.csv`
- `results/crise_maps/synth_saliency_index.csv`
- `results/crise_maps/summary_crise_tau0.1_N1000_s8_p0.5_MASTERSEED123.csv`
- `results/crise_maps/*_saliency_norm.npy`

In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

META_PATH        = "data/synthetic_probes/metadata.csv"
SAL_INDEX_PATH   = "results/crise_maps/synth_saliency_index.csv"
REAL_SUMMARY     = "results/crise_maps/summary_crise_tau0.1_N1000_s8_p0.5_MASTERSEED123.csv"
CRISE_MAP_DIR    = "results/crise_maps"

FIG_DIR          = "results/forensics_figures"

# Primary alpha/strength for per-method comparisons
PRIMARY_ALPHA    = 0.5

# SIM_THRESHOLD is derived empirically in the threshold-derivation cell below.
# Do NOT set it here — it is overwritten by the 5th-pct of real intra-identity
# CRISE cosine similarity.  Value at last run: 0.853
SIM_THRESHOLD    = None   # placeholder; overwritten below

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tqdm import tqdm
from scipy import stats

os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------------
# Facial region masks (fixed in 112×112 aligned chip space)
# Based on _DST_LANDMARKS positions from rise.py:
#   left eye (38,52), right eye (74,52), nose (56,72),
#   mouth-L (42,92), mouth-R (71,92)
# ---------------------------------------------------------------------------

def make_region_masks(h=112, w=112):
    """Return dict of (h,w) float32 binary region masks for the aligned chip."""
    masks = {}
    y, x = np.mgrid[0:h, 0:w]

    def ellipse(cy, cx, ry, rx):
        return (((y - cy) / ry) ** 2 + ((x - cx) / rx) ** 2 <= 1).astype(np.float32)

    # Eye zone: covers both eyes + brow area
    masks["eyes"]     = np.clip(ellipse(50, 38, 18, 20) + ellipse(50, 74, 18, 20), 0, 1)
    # Nose zone
    masks["nose"]     = ellipse(72, 56, 16, 14).astype(np.float32)
    # Mouth zone
    masks["mouth"]    = ellipse(92, 56, 14, 22).astype(np.float32)
    # Forehead / upper face (above eyes)
    masks["forehead"] = ((y < 38) & (x > 15) & (x < 97)).astype(np.float32)
    # Jaw / chin (below mouth)
    masks["jaw_chin"] = ((y > 98) & (x > 15) & (x < 97)).astype(np.float32)

    return masks


REGION_MASKS  = make_region_masks()
REGION_NAMES  = ["forehead", "eyes", "nose", "mouth", "jaw_chin"]

def region_importance(sal_norm):
    """Return dict of fractional saliency weight per region."""
    total = sal_norm.sum() + 1e-12
    return {r: float((sal_norm * REGION_MASKS[r]).sum() / total) for r in REGION_NAMES}


# Quick sanity: visualise masks
fig, axes = plt.subplots(1, len(REGION_NAMES), figsize=(12, 2.5))
for ax, name in zip(axes, REGION_NAMES):
    ax.imshow(REGION_MASKS[name], cmap="viridis", vmin=0, vmax=1)
    ax.set_title(name, fontsize=9)
    ax.axis("off")
plt.suptitle("Region masks (112×112 aligned chip)", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_region_masks.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------

meta      = pd.read_csv(META_PATH)
sal_index = pd.read_csv(SAL_INDEX_PATH)
real_sum  = pd.read_csv(REAL_SUMMARY)

# Normalise column name for real summary
if "true_id" not in real_sum.columns:
    real_sum = real_sum.rename(columns={real_sum.columns[0]: "true_id"})

print(f"Synthetic probes      : {len(meta)}")
print(f"Saliency index entries: {len(sal_index)}")
print(f"Real probe CRISE rows : {len(real_sum)}")

In [ ]:
# ---------------------------------------------------------------------------
# Build per-identity real probe saliency map (mean over all real probes)
# ---------------------------------------------------------------------------

# real_sum must have a 'saliency_path' or we reconstruct from the file naming convention
# Try 'saliency_path' column first; fall back to scanning crise_maps dir

real_sal_by_id = {}   # identity -> mean saliency map (112×112 float32)

if "saliency_path" in real_sum.columns:
    sal_col = "saliency_path"
else:
    # Column may be named differently — find any column ending in .npy path
    npy_cols = [c for c in real_sum.columns if real_sum[c].astype(str).str.endswith(".npy").any()]
    sal_col = npy_cols[0] if npy_cols else None

if sal_col:
    for _, row in tqdm(real_sum.iterrows(), total=len(real_sum), desc="Loading real saliency maps"):
        tid  = row["true_id"]
        path = row[sal_col]
        if not isinstance(path, str) or not os.path.exists(path):
            continue
        sal = np.load(path).astype(np.float32)
        if tid not in real_sal_by_id:
            real_sal_by_id[tid] = []
        real_sal_by_id[tid].append(sal)

    # Mean over all real probes per identity
    real_sal_by_id = {tid: np.mean(maps, axis=0)
                      for tid, maps in real_sal_by_id.items()}
    print(f"Loaded real saliency maps for {len(real_sal_by_id)} identities")
else:
    print("[WARN] No saliency_path column found in real summary — skipping real map load.")
    print("       Update REAL_SUMMARY to point to a CSV that includes saliency_path.")

In [ ]:
# ---------------------------------------------------------------------------
# Principled saliency similarity threshold
#
# A binary "saliency similar / divergent" cutoff must be empirically grounded,
# not arbitrary.  We derive it from real intra-identity CRISE map variation:
#   for every identity with 2+ real probe maps, compute pairwise cosine
#   similarity between those maps.  The threshold is set at the 5th percentile
#   of this distribution.
#
# Interpretation: a synthetic probe's saliency map is "divergent" (Cases B/D)
# only if it falls below the range of variation seen in 95% of genuine same-
# identity real-probe pairs.  This is the smallest defensible threshold —
# anything looser would flag normal cross-probe variation as divergent.
#
# Empirical values (251 identities, 3213 pairs):
#   mean = 0.913   std = 0.032   5th pct = 0.853
# ---------------------------------------------------------------------------

from itertools import combinations
import re

_crise_real_files = [
    f for f in os.listdir(CRISE_MAP_DIR)
    if f.endswith('_saliency_norm.npy') and f.startswith('crise_tau0p1_')
]

# Parse identity — real probes have numeric image IDs like _0001_N; synthetics don't
_id_to_maps = {}
for f in _crise_real_files:
    stem = f.replace('crise_tau0p1_', '').replace('_saliency_norm.npy', '')
    m = re.search(r'^(.+?)_(\w+_\d{4})_N', stem)
    if m:
        _id_to_maps.setdefault(m.group(1), []).append(os.path.join(CRISE_MAP_DIR, f))

_multi = {k: v for k, v in _id_to_maps.items() if len(v) >= 2}
print(f"Identities with 2+ real CRISE maps: {len(_multi)}")

intra_sims = []
for _maps in _multi.values():
    loaded = [np.load(p).ravel().astype(np.float32) for p in _maps]
    for a, b in combinations(loaded, 2):
        an = a / (np.linalg.norm(a) + 1e-9)
        bn = b / (np.linalg.norm(b) + 1e-9)
        intra_sims.append(float(an @ bn))

intra_sims = np.array(intra_sims)
_p5  = float(np.percentile(intra_sims, 5))
_p25 = float(np.percentile(intra_sims, 25))
_med = float(np.median(intra_sims))

print(f"Real intra-identity pairs : {len(intra_sims)}")
print(f"  mean={intra_sims.mean():.3f}  std={intra_sims.std():.3f}")
print(f"  5th pct={_p5:.3f}   25th pct={_p25:.3f}   median={_med:.3f}")

# Overwrite the placeholder
SIM_THRESHOLD = round(_p5, 3)
print(f"
SIM_THRESHOLD set to {SIM_THRESHOLD}  (5th percentile of real intra-identity sim)")

# Cache distribution for the reference figure
np.save(os.path.join(FIG_DIR, "intra_identity_crise_sim.npy"), intra_sims)

# ── Reference figure: real intra-identity sim distribution with threshold line ──
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(intra_sims, bins=60, color='#4E79A7', alpha=0.8, density=True)
ax.axvline(SIM_THRESHOLD, color='#E15759', lw=2, ls='--',
           label=f'5th pct threshold = {SIM_THRESHOLD:.3f}')
ax.axvline(_med, color='#F28E2B', lw=1.5, ls=':',
           label=f'median = {_med:.3f}')
ax.set_xlabel("Cosine similarity (real probe pairs, same identity)", fontsize=10)
ax.set_ylabel("Density", fontsize=10)
ax.set_title("Intra-identity CRISE saliency similarity
"
             "(calibration distribution for SIM_THRESHOLD)", fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
_thresh_fig = os.path.join(FIG_DIR, "fig0_threshold_calibration.png")
plt.savefig(_thresh_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {_thresh_fig}")

In [ ]:
# ---------------------------------------------------------------------------
# Compute saliency divergence for each synthetic probe
# ---------------------------------------------------------------------------

rows_out = []

for _, srow in tqdm(sal_index.iterrows(), total=len(sal_index), desc="Saliency divergence"):
    tid       = srow["identity"]
    sal_path  = srow["saliency_path"]

    if not isinstance(sal_path, str) or not os.path.exists(sal_path):
        continue

    synth_sal = np.load(sal_path).astype(np.float32).ravel()
    synth_sal_2d = synth_sal.reshape(112, 112)

    real_sal_2d  = real_sal_by_id.get(tid)
    if real_sal_2d is None:
        continue

    real_flat = real_sal_2d.ravel()

    # Cosine similarity (both already in [0,1], normalise for dot product)
    s_norm = synth_sal / (np.linalg.norm(synth_sal) + 1e-12)
    r_norm = real_flat / (np.linalg.norm(real_flat) + 1e-12)
    cos_sim = float(np.dot(s_norm, r_norm))
    l1_dist = float(np.abs(synth_sal - real_flat).mean())

    # Per-region importance
    reg_imp = region_importance(synth_sal_2d)

    rows_out.append(dict(
        output_path      = srow["output_path"],
        identity         = tid,
        generation_method= srow["generation_method"],
        blend_alpha      = srow["blend_alpha"],
        saliency_path    = sal_path,
        saliency_cosine_sim = cos_sim,
        saliency_l1         = l1_dist,
        **{f"reg_{k}": v for k, v in reg_imp.items()},
    ))

div_df = pd.DataFrame(rows_out)
print(f"Divergence computed for {len(div_df)} synthetic probes")

In [ ]:
# ---------------------------------------------------------------------------
# Merge divergence back into metadata
# ---------------------------------------------------------------------------

# Drop any stale columns from meta before merging
stale_cols = [c for c in meta.columns
              if c.startswith("reg_") or c in ("saliency_cosine_sim", "saliency_l1", "saliency_path")]
meta_clean = meta.drop(columns=stale_cols)

merge_cols = (["output_path", "saliency_path", "saliency_cosine_sim", "saliency_l1"] +
              [f"reg_{r}" for r in REGION_NAMES])
meta_upd = meta_clean.merge(div_df[merge_cols], on="output_path", how="left")

meta_upd.to_csv(META_PATH, index=False)
print(f"metadata.csv updated → {META_PATH}")
print("Columns added:", [c for c in meta_upd.columns if c.startswith("reg_") or "saliency" in c])

In [ ]:
# ---------------------------------------------------------------------------
# Fig 1: Saliency cosine similarity distribution
# Used to set SIM_THRESHOLD empirically
# ---------------------------------------------------------------------------

sims = div_df["saliency_cosine_sim"].dropna()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(sims, bins=40, color="steelblue", edgecolor="white", linewidth=0.4)
ax.axvline(sims.median(), color="red",    ls="--", lw=1.5, label=f"Median = {sims.median():.3f}")
ax.axvline(sims.quantile(0.25), color="orange", ls=":", lw=1.2,
           label=f"25th pct = {sims.quantile(0.25):.3f}")
ax.set_xlabel("Cosine similarity (synthetic vs real probe saliency maps)")
ax.set_ylabel("Count")
ax.set_title("Saliency map similarity distribution — all synthetic probes")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig1_saliency_sim_dist.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"mean={sims.mean():.3f}  median={sims.median():.3f}  std={sims.std():.3f}")
print(f"Suggested threshold (median): {sims.median():.3f}")
print(">>> Update SIM_THRESHOLD in Config cell if needed, then re-run from the case classification cell.")

In [ ]:
# ---------------------------------------------------------------------------
# Case classification  (update SIM_THRESHOLD above if needed)
# ---------------------------------------------------------------------------

analysis = meta_upd[
    meta_upd["embedding_ok"].astype(bool) &
    meta_upd["rank1_match"].notna() &
    meta_upd["saliency_cosine_sim"].notna()
].copy()

analysis["sal_similar"] = analysis["saliency_cosine_sim"] >= SIM_THRESHOLD

def classify_case(row):
    if row["rank1_match"] and row["sal_similar"]:     return "A"
    if row["rank1_match"] and not row["sal_similar"]: return "B"
    if not row["rank1_match"] and row["sal_similar"]: return "C"
    return "D"

analysis["case_label"] = analysis.apply(classify_case, axis=1)

# Write case labels back to metadata
meta_upd.loc[analysis.index, "case_label"] = analysis["case_label"]
meta_upd.to_csv(META_PATH, index=False)

print(f"Classified {len(analysis)} probes  (threshold={SIM_THRESHOLD})")
print(analysis["case_label"].value_counts().sort_index().to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Fig 2: Cross-method comparison table (Case A/B/C/D % + Rank-1 rate)
# Primary alpha/strength only
# ---------------------------------------------------------------------------

primary = analysis[
    (analysis["blend_alpha"] == PRIMARY_ALPHA) |
    (analysis["generation_method"] == "insightface_swap")  # no alpha for swap
].copy()

rows_table = []
for method, grp in primary.groupby("generation_method"):
    total = len(grp)
    row = {"Method": method, "n": total,
           "Rank-1 rate": grp["rank1_match"].mean()}
    for case in ["A", "B", "C", "D"]:
        row[f"Case {case} %"] = (grp["case_label"] == case).sum() / total * 100
    rows_table.append(row)

table_df = pd.DataFrame(rows_table).set_index("Method")
print("=== Cross-method comparison ===")
print(table_df.round(3).to_string())

# Bar chart
case_cols = ["Case A %", "Case B %", "Case C %", "Case D %"]
colors    = ["#2ecc71", "#e74c3c", "#f39c12", "#95a5a6"]

fig, ax = plt.subplots(figsize=(8, 4))
table_df[case_cols].plot(kind="bar", ax=ax, color=colors, edgecolor="white", width=0.7)
ax.set_ylabel("% of probes")
ax.set_title(f"Case distribution per generation method (α/strength = {PRIMARY_ALPHA})")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
ax.legend(loc="upper right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig2_cross_method_cases.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Fig 3: Per-region importance profiles by case
# ---------------------------------------------------------------------------

reg_cols = [f"reg_{r}" for r in REGION_NAMES]
case_reg = analysis.groupby("case_label")[reg_cols].mean()

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(REGION_NAMES))
width = 0.18
case_colors = {"A": "#2ecc71", "B": "#e74c3c", "C": "#f39c12", "D": "#95a5a6"}

for i, (case, row_) in enumerate(case_reg.iterrows()):
    ax.bar(x + i * width, row_.values, width, label=f"Case {case}",
           color=case_colors.get(case, "gray"), edgecolor="white")

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(REGION_NAMES)
ax.set_ylabel("Mean fractional saliency weight")
ax.set_title("Per-region importance by forensic case")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig3_region_importance_by_case.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\nMean per-region importance by case:")
print(case_reg.round(4).to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Fig 4: Saliency similarity vs ArcFace similarity (scatter, coloured by case)
# ---------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6, 5))
for case, grp in analysis.groupby("case_label"):
    ax.scatter(grp["arcface_similarity"], grp["saliency_cosine_sim"],
               label=f"Case {case}", alpha=0.5, s=18,
               color=case_colors.get(case, "gray"))

ax.axhline(SIM_THRESHOLD, color="black", ls="--", lw=1, label=f"Threshold={SIM_THRESHOLD:.2f}")
ax.set_xlabel("ArcFace cosine similarity to true identity")
ax.set_ylabel("Saliency map cosine similarity (synthetic vs real)")
ax.set_title("ArcFace confidence vs saliency faithfulness")
ax.legend(fontsize=8, markerscale=1.5)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig4_sim_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Fig 5 & 6: Alpha/strength ablation — rank-1 rate and case distribution
# ---------------------------------------------------------------------------

for method in analysis["generation_method"].unique():
    grp_m = analysis[analysis["generation_method"] == method]
    if grp_m["blend_alpha"].nunique() < 2:
        continue

    ablation_rows = []
    for a, grp_a in grp_m.groupby("blend_alpha"):
        r = {"alpha": a, "rank1_rate": grp_a["rank1_match"].mean()}
        for case in ["A", "B", "C", "D"]:
            r[f"case_{case}"] = (grp_a["case_label"] == case).mean() * 100
        ablation_rows.append(r)
    abl_df = pd.DataFrame(ablation_rows).sort_values("alpha")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))

    ax1.plot(abl_df["alpha"], abl_df["rank1_rate"], "o-", color="steelblue")
    ax1.set_xlabel("alpha / strength")
    ax1.set_ylabel("Rank-1 rate")
    ax1.set_title(f"{method} — fooling rate vs alpha")
    ax1.grid(alpha=0.3)

    for case, color in case_colors.items():
        ax2.plot(abl_df["alpha"], abl_df[f"case_{case}"], "o-",
                 color=color, label=f"Case {case}")
    ax2.set_xlabel("alpha / strength")
    ax2.set_ylabel("% of probes")
    ax2.set_title(f"{method} — case distribution vs alpha")
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)

    plt.suptitle(f"Ablation: {method}", fontsize=11)
    plt.tight_layout()
    fname = f"fig_ablation_{method}.png"
    plt.savefig(os.path.join(FIG_DIR, fname), dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Figs 7+: Example saliency map pairs — 2 examples per case (8 figures)
# Shows: real probe saliency | synthetic saliency | overlay difference
# ---------------------------------------------------------------------------

N_EXAMPLES = 2

for case in ["A", "B", "C", "D"]:
    case_rows = analysis[analysis["case_label"] == case].dropna(
        subset=["saliency_path"]
    ).head(N_EXAMPLES)

    if len(case_rows) == 0:
        print(f"No examples for Case {case}")
        continue

    fig, axes = plt.subplots(len(case_rows), 3,
                             figsize=(9, 3 * len(case_rows)))
    if len(case_rows) == 1:
        axes = axes[np.newaxis, :]

    for row_i, (_, row_) in enumerate(case_rows.iterrows()):
        tid = row_["identity"]

        synth_sal = np.load(row_["saliency_path"]).astype(np.float32)
        real_sal  = real_sal_by_id.get(tid)

        diff = np.abs(synth_sal - real_sal) if real_sal is not None else np.zeros_like(synth_sal)

        titles = [
            f"Real probe\n{tid[:22]}",
            f"Synthetic ({row_['generation_method']})\ncos_sim={row_['saliency_cosine_sim']:.3f}",
            f"|Δ saliency|\nrank1={row_['rank1_match']}  case={case}",
        ]
        imgs = [real_sal if real_sal is not None else np.zeros_like(synth_sal),
                synth_sal, diff]

        for col_i, (img, title) in enumerate(zip(imgs, titles)):
            axes[row_i, col_i].imshow(img, cmap="hot", vmin=0, vmax=1)
            axes[row_i, col_i].set_title(title, fontsize=8)
            axes[row_i, col_i].axis("off")

    plt.suptitle(f"Case {case} examples", fontsize=11)
    plt.tight_layout()
    fname = f"fig_examples_case{case}.png"
    plt.savefig(os.path.join(FIG_DIR, fname), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Case {case}: saved {fname}")

In [ ]:
# ---------------------------------------------------------------------------
# Summary table (paper-ready)
# ---------------------------------------------------------------------------

print("=== Full cross-method summary (all alphas/strengths) ===")
full_table = []
for method, grp in analysis.groupby("generation_method"):
    total = len(grp)
    r = {"Method": method, "n": total,
         "Rank-1 rate": f"{grp['rank1_match'].mean():.3f}"}
    for case in ["A", "B", "C", "D"]:
        pct = (grp["case_label"] == case).sum() / total * 100
        r[f"Case {case} %"] = f"{pct:.1f}"
    full_table.append(r)

print(pd.DataFrame(full_table).set_index("Method").to_string())

print("\n=== Primary alpha only ===")
print(table_df.round(3).to_string())

print(f"\nAll figures saved to: {FIG_DIR}/")

---
## Fig 8 — Case A Deep Dive: Real vs. Synthetic Individual CRISE Comparison

The existing saliency divergence metric compares each synthetic map to the **mean** real-probe map per identity. This cell uses **individual probe pairs** — directly matching each Case A synthetic to the specific real probe it is paired with. This is what a court would actually want: was ArcFace looking at the same facial evidence for both the real photo and the deepfake?

Four examples are shown: the 2 most similar and 2 most divergent Case A pairs by cosine similarity.

In [ ]:
# ---------------------------------------------------------------------------
# Fig 8 — Case A deep dive: individual real vs. synthetic CRISE comparison
# ---------------------------------------------------------------------------

case_a_rows = analysis[analysis["case_label"] == "A"].copy()
print(f"Case A probes: {len(case_a_rows)}")

# Build identity -> list of (probe_path, sal_path) from real_sum
real_probe_paths = {}
for _, row in real_sum.iterrows():
    sp = row.get("saliency_path", None)
    if isinstance(sp, str) and os.path.exists(sp):
        real_probe_paths.setdefault(row["true_id"], []).append(
            (row["probe_path"], sp)
        )

pair_records = []
for _, row in case_a_rows.iterrows():
    identity  = row["identity"]
    synth_sal = row.get("saliency_path") if isinstance(row.get("saliency_path"), str) else None
    if synth_sal is None or not os.path.exists(synth_sal):
        continue
    real_options = real_probe_paths.get(identity, [])
    if not real_options:
        continue

    synth_map = np.load(synth_sal).astype(np.float32)
    sv = synth_map.ravel(); sv /= (np.linalg.norm(sv) + 1e-9)

    real_ppath, real_sal_path = real_options[0]
    real_map = np.load(real_sal_path).astype(np.float32)
    rv = real_map.ravel(); rv /= (np.linalg.norm(rv) + 1e-9)

    indiv_cos = float(rv @ sv)

    reg_div = {}
    for rname, mask in REGION_MASKS.items():
        r_frac = float(real_map[mask].sum() / (real_map.sum() + 1e-9))
        s_frac = float(synth_map[mask].sum() / (synth_map.sum() + 1e-9))
        reg_div[rname] = abs(r_frac - s_frac)

    pair_records.append({
        "identity":        identity,
        "synth_path":      row["output_path"],
        "real_probe_path": real_ppath,
        "real_sal_path":   real_sal_path,
        "synth_sal_path":  synth_sal,
        "indiv_cos":       indiv_cos,
        "method":          row["generation_method"],
        "blend_alpha":     row["blend_alpha"],
        "arcface_sim":     row["arcface_similarity"],
        **{f"reg_div_{k}": v for k, v in reg_div.items()},
    })

pair_df = pd.DataFrame(pair_records).sort_values("indiv_cos", ascending=False)
print(f"Pairs evaluated: {len(pair_df)}")
print(f"  cos sim — mean={pair_df['indiv_cos'].mean():.3f}  "
      f"std={pair_df['indiv_cos'].std():.3f}  min={pair_df['indiv_cos'].min():.3f}")

reg_div_cols = [c for c in pair_df.columns if c.startswith("reg_div_")]
print("\nMean per-region |divergence| across Case A individual pairs:")
for c in reg_div_cols:
    print(f"  {c.replace('reg_div_', ''):12s}  {pair_df[c].mean():.4f}")

pair_df.to_csv(os.path.join(FIG_DIR, "case_a_individual_pairs.csv"), index=False)

# ── Visualise 2 best + 2 worst Case A pairs ─────────────────────────────
top2    = pair_df.head(2)
bottom2 = pair_df.tail(2)
showcase = pd.concat([top2, bottom2], ignore_index=True)
row_labels = ["Best (1)", "Best (2)", "Worst (1)", "Worst (2)"]

from insightface.app import FaceAnalysis as _FA
_app_fig8 = _FA(name="buffalo_l",
                providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
_app_fig8.prepare(ctx_id=0, det_size=(640, 640))

fig, axes = plt.subplots(4, 4, figsize=(13, 13))
col_titles = ["Real chip", "Real CRISE", "Synth chip", "Synth CRISE"]
for j, t in enumerate(col_titles):
    axes[0, j].set_title(t, fontsize=10, fontweight="bold")

for i, (_, pr) in enumerate(showcase.iterrows()):
    real_img  = cv2.imread(pr["real_probe_path"])
    synth_img = cv2.imread(pr["synth_path"])
    real_map  = np.load(pr["real_sal_path"]).astype(np.float32)
    synth_map = np.load(pr["synth_sal_path"]).astype(np.float32)

    def _chip(img):
        from rise import build_aligned_chip_112
        if img is None:
            return np.zeros((112, 112, 3), dtype=np.uint8)
        try:
            return build_aligned_chip_112(img, _app_fig8)
        except Exception:
            return np.zeros((112, 112, 3), dtype=np.uint8)

    rc = _chip(real_img);  sc = _chip(synth_img)
    axes[i, 0].imshow(cv2.cvtColor(rc, cv2.COLOR_BGR2RGB))
    axes[i, 1].imshow(real_map,  cmap="inferno", vmin=0, vmax=1)
    axes[i, 2].imshow(cv2.cvtColor(sc, cv2.COLOR_BGR2RGB))
    axes[i, 3].imshow(synth_map, cmap="inferno", vmin=0, vmax=1)

    title_color = "#2ecc71" if i < 2 else "#e74c3c"
    axes[i, 0].set_ylabel(
        f"{row_labels[i]}\n{pr['identity']}\n({pr['method']})",
        fontsize=7, rotation=0, labelpad=90, va="center")
    axes[i, 1].set_title(f"arcface={pr['arcface_sim']:.3f}", fontsize=8)
    axes[i, 3].set_title(f"cos={pr['indiv_cos']:.3f}", fontsize=9,
                          color=title_color, fontweight="bold")

for ax in axes.ravel():
    ax.axis("off")

plt.suptitle(
    "Fig 8 — Case A: Real vs. Synthetic CRISE maps\n"
    "Top 2: most similar pairs  |  Bottom 2: most divergent pairs",
    fontsize=12, y=1.01)
plt.tight_layout()
_fig8 = os.path.join(FIG_DIR, "fig8_case_a_individual_pairs.png")
plt.savefig(_fig8, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {_fig8}")


---
## Fig 9 — Identity Vulnerability Analysis

Which identities are structurally more at risk from morphing and SD img2img attacks?  
Each target identity has 3 synthetic probes per method. We classify identities as  
**vulnerable** (all 3 fooled), **partial** (1–2 fooled), or **resistant** (0 fooled).

This is the societal-impact finding made concrete: CRISE-ID identifies *which individuals* face greater forensic risk from deepfake attacks. Connect to `demographic_analysis.ipynb` to test whether vulnerability correlates with estimated gender or age bracket.

In [ ]:
# ---------------------------------------------------------------------------
# Fig 9 — Identity vulnerability analysis
# ---------------------------------------------------------------------------

vuln_rows = []
for identity, grp in meta.groupby("identity"):
    for method, mgrp in grp.groupby("generation_method"):
        if method == "insightface_swap":
            primary = mgrp
        else:
            primary = mgrp[mgrp["blend_alpha"] == 0.5]
        if primary.empty:
            continue
        r1 = int(primary["rank1_match"].sum())
        n  = len(primary)
        if   r1 == n:  status = "Vulnerable"
        elif r1 == 0:  status = "Resistant"
        else:          status = "Partial"
        vuln_rows.append({"identity": identity, "method": method,
                          "n_probes": n, "n_fooled": r1,
                          "status": status, "fool_rate": r1 / n})

vuln_df = pd.DataFrame(vuln_rows)
print("Vulnerability distribution (α=0.5 / all for swap):")
print(vuln_df.groupby(["method", "status"]).size().unstack(fill_value=0))
vuln_df.to_csv(os.path.join(FIG_DIR, "identity_vulnerability.csv"), index=False)

# ── Figure ────────────────────────────────────────────────────────────────
methods_plot  = [m for m in ["morphing", "sd_img2img", "insightface_swap"]
                 if m in vuln_df["method"].unique()]
status_colors = {"Vulnerable": "#e74c3c", "Partial": "#f39c12", "Resistant": "#2ecc71"}
status_order  = ["Vulnerable", "Partial", "Resistant"]

fig, axes = plt.subplots(1, len(methods_plot),
                          figsize=(4.5 * len(methods_plot), 4.5))
if len(methods_plot) == 1:
    axes = [axes]

for ax, method in zip(axes, methods_plot):
    sub   = vuln_df[vuln_df["method"] == method]
    total = len(sub)
    bars  = [sub["status"].value_counts().get(s, 0) for s in status_order]
    pcts  = [100 * b / total for b in bars]
    ax.bar(status_order, pcts,
           color=[status_colors[s] for s in status_order],
           alpha=0.85, edgecolor="white", linewidth=0.8)
    for s, p, b in zip(status_order, pcts, bars):
        ax.text(s, p + 0.5, str(b), ha="center", va="bottom", fontsize=10)
    ax.set_title(method.replace("_", " ").title(), fontsize=11)
    ax.set_ylabel("% of target identities", fontsize=9)
    ax.set_ylim(0, max(pcts) * 1.2 + 5)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle(
    "Fig 9 — Identity vulnerability to synthetic attacks  (primary α / strength = 0.5)",
    fontsize=12)
plt.tight_layout()
_fig9 = os.path.join(FIG_DIR, "fig9_identity_vulnerability.png")
plt.savefig(_fig9, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {_fig9}")

morph_vuln = vuln_df[vuln_df["method"] == "morphing"].sort_values("fool_rate", ascending=False)
print("\nTop 10 most vulnerable identities (morphing α=0.5):")
print(morph_vuln.head(10)[["identity", "n_fooled", "n_probes", "status"]].to_string(index=False))
print("\n10 most resistant identities:")
print(morph_vuln.tail(10)[["identity", "n_fooled", "n_probes", "status"]].to_string(index=False))


---
## Fig 10 — Identity Absorption Curve

The central novel visual: saliency cosine similarity (line) and rank-1 success rate (bars) as a function of generation strength, for both attack types.

- **Morphing (left):** monotonic — more target identity blended → higher saliency sim → higher rank-1
- **SD img2img (right):** *inverted* — low strength barely alters the original (trivially fools ArcFace); high strength generates a new AI face that ArcFace rejects. Same model, opposite forensic outcome.

In [ ]:
# ---------------------------------------------------------------------------
# Fig 10 — Identity absorption curve
# Dual-axis per method: rank-1 rate (bars) + mean saliency cosine sim (line ± 1 std)
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
method_colors = {"morphing": "#4E79A7", "sd_img2img": "#E15759"}

for ax, method in zip(axes, ["morphing", "sd_img2img"]):
    sub    = meta[(meta["generation_method"] == method) &
                  meta["saliency_cosine_sim"].notna()].copy()
    alphas = sorted(sub["blend_alpha"].dropna().unique())
    r1_pct = [sub[sub["blend_alpha"] == a]["rank1_match"].mean() * 100 for a in alphas]
    cs_avg = [sub[sub["blend_alpha"] == a]["saliency_cosine_sim"].mean() for a in alphas]
    cs_std = [sub[sub["blend_alpha"] == a]["saliency_cosine_sim"].std()  for a in alphas]

    color = method_colors[method]
    x = np.arange(len(alphas))
    ax2 = ax.twinx()

    bars = ax.bar(x, r1_pct, width=0.4, alpha=0.55, color=color, label="Rank-1 rate (%)")
    line, = ax2.plot(x, cs_avg, "o-", color="black", lw=2.5, label="Saliency cos sim", zorder=5)
    ax2.fill_between(x, [c - s for c, s in zip(cs_avg, cs_std)],
                         [c + s for c, s in zip(cs_avg, cs_std)],
                     color="black", alpha=0.1)

    ax.set_xticks(x)
    ax.set_xticklabels([str(a) for a in alphas], fontsize=12)
    xlabel = "Blend ratio α (morphing)" if method == "morphing" else "Generation strength"
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel("Rank-1 success rate (%)", fontsize=10, color=color)
    ax.tick_params(axis="y", labelcolor=color)
    ax.set_ylim(0, 115)
    ax2.set_ylabel("Mean saliency cosine similarity", fontsize=10)
    ax2.set_ylim(0.80, 0.98)

    title = ("Morphing — identity absorption\n"
             "(higher α → more identity transferred)")
    if method == "sd_img2img":
        title = ("SD img2img — behavioral inversion\n"
                 "(higher strength → new AI face → lower rank-1)")
    ax.set_title(title, fontsize=10)
    ax.grid(axis="y", alpha=0.25)

    legend_loc = "upper left" if method == "morphing" else "upper right"
    ax.legend([bars, line], ["Rank-1 rate (%)", "Mean saliency cos sim ± 1 std"],
              loc=legend_loc, fontsize=8, framealpha=0.9)

plt.suptitle("Fig 10 — Identity absorption curve: saliency similarity tracks identity transfer",
             fontsize=12, y=1.02)
plt.tight_layout()
_fig10 = os.path.join(FIG_DIR, "fig10_identity_absorption_curve.png")
plt.savefig(_fig10, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {_fig10}")

for method in ["morphing", "sd_img2img"]:
    sub = meta[(meta["generation_method"] == method) & meta["saliency_cosine_sim"].notna()]
    for a in sorted(sub["blend_alpha"].dropna().unique()):
        g = sub[sub["blend_alpha"] == a]
        print(f"  [{method}] strength/alpha={a}  "
              f"rank1={g['rank1_match'].mean()*100:.0f}%  "
              f"sal_cos={g['saliency_cosine_sim'].mean():.3f}±{g['saliency_cosine_sim'].std():.3f}")


---
## Fig 11 — RISE vs. CRISE: Identity Tracking on Synthetic Probes

CRISE outperforms RISE on real probes (insertion/deletion AUC). This section extends that to the forensic setting: for Case A synthetic probes (deepfakes that fool ArcFace), is the real probe's **CRISE** map more similar to the synthetic's CRISE map than the real probe's **RISE** map is?

Metric: `cos(real_CRISE, synth_CRISE)` vs `cos(real_RISE, synth_CRISE)`.  
If CRISE-CRISE > RISE-CRISE, CRISE better captures the identity evidence the deepfake is exploiting — a forensic advantage unavailable to baseline RISE.

Statistical test: Wilcoxon signed-rank (paired, non-parametric).

In [ ]:
# ---------------------------------------------------------------------------
# Fig 11 — RISE vs. CRISE: which method better tracks synthetic identity transfer?
# ---------------------------------------------------------------------------

RISE_SUMMARY_PATH = "results/rise_baseline/summary_K1680_N1000_s8_p0.5_MASTERSEED123.csv"
rise_sum_df = pd.read_csv(RISE_SUMMARY_PATH)
rise_sum_df = rise_sum_df[
    rise_sum_df["saliency_path"].notna() &
    rise_sum_df["saliency_path"].apply(os.path.exists)
].copy()

rise_real_sal = {row["true_id"]: row["saliency_path"]
                 for _, row in rise_sum_df.iterrows()}

from scipy.stats import wilcoxon

records_rc = []
for _, row in analysis[analysis["case_label"] == "A"].iterrows():
    identity  = row["identity"]
    synth_sal = row.get("saliency_path") if isinstance(row.get("saliency_path"), str) else None
    if synth_sal is None or not os.path.exists(synth_sal):
        continue
    real_crise_opts = real_probe_paths.get(identity, [])
    if not real_crise_opts:
        continue
    real_rise_path = rise_real_sal.get(identity)
    if real_rise_path is None or not os.path.exists(real_rise_path):
        continue

    try:
        def _nload(p):
            v = np.load(p).ravel().astype(np.float32)
            return v / (np.linalg.norm(v) + 1e-9)

        sv  = _nload(synth_sal)
        cv  = _nload(real_crise_opts[0][1])
        rv  = _nload(real_rise_path)

        records_rc.append({
            "identity":        identity,
            "method":          row["generation_method"],
            "blend_alpha":     row["blend_alpha"],
            "cos_crise_synth": float(cv @ sv),
            "cos_rise_synth":  float(rv @ sv),
            "cos_crise_rise":  float(cv @ rv),
            "crise_advantage": float(cv @ sv) - float(rv @ sv),
            "arcface_sim":     row["arcface_similarity"],
        })
    except Exception:
        pass

rc_df = pd.DataFrame(records_rc)
print(f"Case A pairs with both RISE + CRISE real maps: {len(rc_df)}")

if len(rc_df) > 0:
    m_c = rc_df["cos_crise_synth"].mean()
    m_r = rc_df["cos_rise_synth"].mean()
    adv = rc_df["crise_advantage"].mean()
    stat, pval = wilcoxon(rc_df["cos_crise_synth"], rc_df["cos_rise_synth"])
    print(f"  mean cos(real_CRISE, synth) = {m_c:.4f}")
    print(f"  mean cos(real_RISE,  synth) = {m_r:.4f}")
    print(f"  CRISE advantage             = {adv:+.4f}")
    print(f"  Wilcoxon: stat={stat:.1f}, p={pval:.4f}")
    rc_df.to_csv(os.path.join(FIG_DIR, "rise_vs_crise_on_synthetics.csv"), index=False)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    # Panel 1: paired scatter
    ax = axes[0]
    sc = ax.scatter(rc_df["cos_rise_synth"], rc_df["cos_crise_synth"],
                    alpha=0.5, s=20, c=rc_df["arcface_sim"], cmap="viridis")
    plt.colorbar(sc, ax=ax, label="ArcFace similarity")
    lim_lo = min(rc_df[["cos_rise_synth", "cos_crise_synth"]].min()) - 0.01
    lim_hi = max(rc_df[["cos_rise_synth", "cos_crise_synth"]].max()) + 0.01
    ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], "k--", lw=1, label="Equal")
    frac_above = (rc_df["crise_advantage"] > 0).mean()
    ax.set_xlabel("cos(real RISE, synth CRISE)", fontsize=10)
    ax.set_ylabel("cos(real CRISE, synth CRISE)", fontsize=10)
    ax.set_title(f"Points above diagonal: CRISE tracks better\n({frac_above*100:.0f}% of pairs)", fontsize=9)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    # Panel 2: advantage histogram
    ax2 = axes[1]
    ax2.hist(rc_df["crise_advantage"], bins=30, color="#4E79A7", alpha=0.8, edgecolor="white")
    ax2.axvline(0,   color="k",       lw=1.5, ls="--", label="No difference")
    ax2.axvline(adv, color="#E15759", lw=2,
                label=f"mean={adv:+.4f}  (p={pval:.4f})")
    ax2.set_xlabel("CRISE advantage  (cos_CRISE − cos_RISE)", fontsize=10)
    ax2.set_ylabel("Count", fontsize=10)
    ax2.set_title("CRISE advantage distribution across Case A pairs", fontsize=10)
    ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

    plt.suptitle("Fig 11 — RISE vs. CRISE: identity tracking on Case A synthetic probes",
                 fontsize=12)
    plt.tight_layout()
    _fig11 = os.path.join(FIG_DIR, "fig11_rise_vs_crise_synthetics.png")
    plt.savefig(_fig11, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {_fig11}")
else:
    print("No matched pairs — check real_probe_paths and rise_real_sal are populated.")


---
## Auditor-Oriented Forensic Analysis

The taxonomy analysis above (Cases A–D) characterises the *population* of synthetic probes.
This section reframes those findings as **per-identity forensic evidence** that a human auditor
can inspect directly. For a selected identity it answers:

1. **Where is the model looking?** — which facial regions carry the most saliency weight on real probes?
2. **Is evidence consistent across real images?** — how stable is the saliency map for the same person?
3. **Does synthetic generation alter what evidence is used?** — does the model shift attention when
   looking at generated images?
4. **Does saliency drift toward artifact or non-identity regions?** — hairline, background, cheeks?
5. **Is high confidence supported by believable evidence?** — are there synthetic matches with
   high ArcFace similarity but suspicious saliency patterns?

> **Framing:** When you enrol in a face recognition system, the system knows which parts of your
> face it *owns*. You don't. CRISE-ID closes that gap.

Helper functions are **method-agnostic** — they accept raw saliency arrays and a label, so the
same cells work for RISE or CRISE by changing one path variable.

In [ ]:
# =============================================================================
# Cell 27 — Extended region masks (v2) + saliency helper functions
# =============================================================================
import numpy as np
from scipy import stats as _scipy_stats

# --- Extended region masks (keeps existing REGION_MASKS intact) ---

def make_region_masks_v2(h=112, w=112):
    """7-region mask dict for 112x112 aligned chip.

    Regions:
      forehead   — upper face above eyes
      eyes       — bilateral eye ellipses
      nose       — nose-bridge ellipse
      mouth      — mouth ellipse
      jaw_chin   — lower chin strip
      cheeks_jaw — lateral cheek bands (outside core face)
      background — outer border / non-face corners
    """
    y, x = np.mgrid[0:h, 0:w]

    def ellipse(cy, cx, ry, rx):
        return (((y - cy) / ry) ** 2 + ((x - cx) / rx) ** 2 <= 1).astype(np.float32)

    masks = {}
    masks["forehead"]   = ((y < 38) & (x > 15) & (x < 97)).astype(np.float32)
    masks["eyes"]       = np.clip(ellipse(50, 38, 18, 20) + ellipse(50, 74, 18, 20), 0, 1)
    masks["nose"]       = ellipse(72, 56, 16, 14)
    masks["mouth"]      = ellipse(92, 56, 14, 22)
    masks["jaw_chin"]   = ((y > 98) & (x > 15) & (x < 97)).astype(np.float32)
    # Lateral cheek bands — flanking the core face oval
    masks["cheeks_jaw"] = (((x < 28) | (x > 84)) & (y > 55) & (y < 105)).astype(np.float32)
    # Background — pixels outside a broad face oval
    core = np.clip(ellipse(56, 56, 56, 48), 0, 1)
    masks["background"] = (1.0 - core).astype(np.float32)
    return masks

REGION_MASKS_V2 = make_region_masks_v2()
REGION_NAMES_V2 = ["forehead", "eyes", "nose", "mouth", "jaw_chin", "cheeks_jaw", "background"]

def region_importance_v2(sal_norm):
    """Fractional saliency weight per region (7-region v2)."""
    total = sal_norm.sum() + 1e-12
    return {r: float((sal_norm * REGION_MASKS_V2[r]).sum() / total) for r in REGION_NAMES_V2}


# --- Saliency helper functions (method-agnostic) ---

def top_k_overlap(sal_a, sal_b, k=0.10):
    """Top-k% pixel IoU between two saliency maps.
    Returns dict: {iou, intersection_frac}.
    """
    a = sal_a.ravel().astype(np.float32)
    b = sal_b.ravel().astype(np.float32)
    n_top = max(1, int(len(a) * k))
    thresh_a = np.partition(a, -n_top)[-n_top]
    thresh_b = np.partition(b, -n_top)[-n_top]
    mask_a = a >= thresh_a
    mask_b = b >= thresh_b
    intersection = float((mask_a & mask_b).sum())
    union        = float((mask_a | mask_b).sum())
    return {"iou": intersection / (union + 1e-9),
            "intersection_frac": intersection / (n_top + 1e-9)}


def saliency_correlation(sal_a, sal_b):
    """Pearson and Spearman correlation. Returns dict: {pearson, spearman}."""
    a = sal_a.ravel().astype(np.float64)
    b = sal_b.ravel().astype(np.float64)
    if a.std() < 1e-9 or b.std() < 1e-9:
        return {"pearson": 0.0, "spearman": 0.0}
    pr, _ = _scipy_stats.pearsonr(a, b)
    sr, _ = _scipy_stats.spearmanr(a, b)
    return {"pearson": float(pr), "spearman": float(sr)}


def saliency_concentration(sal, k=0.10):
    """Fraction of total saliency mass in the top-k% pixels.
    Uniform map → k; perfectly concentrated → 1.0.
    """
    a = sal.ravel().astype(np.float32)
    total = a.sum()
    if total < 1e-12:
        return 0.0
    n_top = max(1, int(len(a) * k))
    return float(np.partition(a, -n_top)[-n_top:].sum() / total)


def saliency_entropy(sal):
    """Normalised Shannon entropy. 1.0 = uniform, 0.0 = single pixel."""
    a = sal.ravel().astype(np.float64)
    total = a.sum()
    if total < 1e-12:
        return 1.0
    p = a / total
    p = p[p > 0]
    return float(-np.sum(p * np.log(p)) / np.log(len(a)))


def cosine_sim_maps(sal_a, sal_b):
    """Cosine similarity between two saliency maps."""
    a = sal_a.ravel().astype(np.float32)
    b = sal_b.ravel().astype(np.float32)
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    if na < 1e-9 or nb < 1e-9:
        return 0.0
    return float(np.dot(a / na, b / nb))


# Mismatch flag — thresholds are module-level so callers can patch them easily.
_MISMATCH_CONC_THRESH = 0.35     # low concentration
_MISMATCH_BG_THRESH   = 0.08     # high background mass
# _MISMATCH_HIGH_CONF_THRESH is set in Cell 29 from data percentile

def flag_confidence_evidence_mismatch(
    arcface_sim, sal_conc, cos_sim_vs_real, bg_mass,
    high_conf_thresh=None,
    conc_thresh=_MISMATCH_CONC_THRESH,
    cosim_thresh=None,
    bg_thresh=_MISMATCH_BG_THRESH,
):
    """Return True when ArcFace confidence is high but saliency evidence is suspicious.

    High confidence = arcface_sim >= high_conf_thresh (75th pct of synthetic sims).
    Suspicious = low concentration OR low real-reference cosine sim OR high background.
    """
    if cosim_thresh is None:
        cosim_thresh = SIM_THRESHOLD
    if high_conf_thresh is None or arcface_sim is None:
        return False
    try:
        if np.isnan(float(arcface_sim)):
            return False
    except (TypeError, ValueError):
        return False
    if float(arcface_sim) < float(high_conf_thresh):
        return False
    suspicious = (
        (sal_conc is not None and not np.isnan(float(sal_conc)) and float(sal_conc) < conc_thresh) or
        (cos_sim_vs_real is not None and not np.isnan(float(cos_sim_vs_real))
             and float(cos_sim_vs_real) < cosim_thresh) or
        (bg_mass is not None and not np.isnan(float(bg_mass)) and float(bg_mass) > bg_thresh)
    )
    return bool(suspicious)


print("Cell 27: extended masks + helper functions defined.")
print(f"  Regions: {REGION_NAMES_V2}")

In [ ]:
# =============================================================================
# Cell 28 — Build forensic_df: unified real + synthetic per-image dataframe
# =============================================================================
import os
import pandas as pd

# Identities that have synthetic probes (the ~50 target identities)
synth_identities = set(analysis["identity"].unique())

# ── Real rows ──────────────────────────────────────────────────────────────
real_rows = []
_real_skip = 0

for _, row in real_sum.iterrows():
    tid = row["true_id"]
    if tid not in synth_identities:
        continue  # only include identities that also have synthetic probes
    sal_path = row.get("saliency_path", None)
    if not isinstance(sal_path, str) or not os.path.exists(sal_path):
        _real_skip += 1
        continue
    sal = np.load(sal_path).astype(np.float32)
    reg = region_importance_v2(sal)
    real_rows.append({
        "identity":          tid,
        "image_path":        row.get("probe_path", ""),
        "is_synthetic":      False,
        "generation_method": "real",
        "blend_alpha":       None,
        "rank1_match":       True,   # real probes were used to build gallery
        "arcface_similarity":None,   # not applicable for real probe vs self
        "case_label":        "real",
        "saliency_path":     sal_path,
        "saliency_map":      sal,
        "sal_concentration": saliency_concentration(sal),
        "sal_entropy":       saliency_entropy(sal),
        "sal_valid":         True,
        **{f"reg_{r}_v2": v for r, v in reg.items()},
    })

print(f"Real rows loaded   : {len(real_rows)}  (skipped {_real_skip} missing saliency files)")

# ── Synthetic rows ──────────────────────────────────────────────────────────
synth_rows = []
_synth_skip = 0

for _, row in analysis.iterrows():
    sal_path = row.get("saliency_path", None)
    if not isinstance(sal_path, str) or not os.path.exists(sal_path):
        _synth_skip += 1
        continue
    sal = np.load(sal_path).astype(np.float32)
    reg = region_importance_v2(sal)
    synth_rows.append({
        "identity":          row["identity"],
        "image_path":        row.get("output_path", ""),
        "is_synthetic":      True,
        "generation_method": row.get("generation_method", ""),
        "blend_alpha":       row.get("blend_alpha", None),
        "rank1_match":       row.get("rank1_match", None),
        "arcface_similarity":row.get("arcface_similarity", None),
        "case_label":        row.get("case_label", "?"),
        "saliency_path":     sal_path,
        "saliency_map":      sal,
        "sal_concentration": saliency_concentration(sal),
        "sal_entropy":       saliency_entropy(sal),
        "sal_valid":         True,
        **{f"reg_{r}_v2": v for r, v in reg.items()},
    })

print(f"Synthetic rows loaded: {len(synth_rows)}  (skipped {_synth_skip} missing)")

forensic_df = pd.DataFrame(real_rows + synth_rows).reset_index(drop=True)
print(f"forensic_df shape  : {forensic_df.shape}")
print(f"Identities covered : {forensic_df['identity'].nunique()}")
print(f"Columns            : {list(forensic_df.columns)}")

In [ ]:
# =============================================================================
# Cell 29 — Select demo identity + compute per-probe metrics vs real reference
# =============================================================================

# ── Select DEMO_IDENTITY ────────────────────────────────────────────────────
# Criteria: valid real saliency in real_sal_by_id, >=2 synthetic rows,
# and ideally has both Case A and Case B examples.

_candidates = []
for _id in forensic_df["identity"].unique():
    _sub = forensic_df[forensic_df["identity"] == _id]
    _real = _sub[~_sub["is_synthetic"]]
    _synth = _sub[_sub["is_synthetic"]]
    if _id not in real_sal_by_id or len(_real) == 0 or len(_synth) < 2:
        continue
    _has_A = ("A" in _synth["case_label"].values)
    _has_B = ("B" in _synth["case_label"].values)
    _candidates.append((_id, _has_A and _has_B, len(_synth)))

_candidates.sort(key=lambda x: (not x[1], -x[2]))  # prefer A+B, then most synthetics

if not _candidates:
    raise RuntimeError("No suitable demo identity found — check forensic_df build.")

DEMO_IDENTITY = _candidates[0][0]
print(f"Selected DEMO_IDENTITY: {DEMO_IDENTITY}")
print(f"  Has Case A + B: {_candidates[0][1]}")
print(f"  Synthetic rows: {_candidates[0][2]}")

# ── Compute high-confidence threshold (75th pct of synthetic ArcFace sims) ─
_arc_sims = forensic_df.loc[
    forensic_df["is_synthetic"] & forensic_df["arcface_similarity"].notna(),
    "arcface_similarity"
].astype(float)
_HIGH_CONF_THRESH = float(np.percentile(_arc_sims, 75)) if len(_arc_sims) > 0 else 0.5
print(f"\nHigh-confidence threshold (75th pct ArcFace sim): {_HIGH_CONF_THRESH:.4f}")

# ── Add per-probe metrics vs real reference ──────────────────────────────────
# For each row compute: cosine sim to real reference, top-k overlap, Pearson, Spearman
# and the confidence-evidence mismatch flag.

_cosim_col   = []
_iou_col     = []
_pearson_col = []
_spearman_col= []
_mismatch_col= []

for _, row in forensic_df.iterrows():
    _tid = row["identity"]
    _real_ref = real_sal_by_id.get(_tid)
    _sal = row["saliency_map"]
    if _real_ref is None or _sal is None or not row["sal_valid"]:
        _cosim_col.append(None)
        _iou_col.append(None)
        _pearson_col.append(None)
        _spearman_col.append(None)
        _mismatch_col.append(False)
        continue
    _cs = cosine_sim_maps(_sal, _real_ref)
    _tk = top_k_overlap(_sal, _real_ref, k=0.10)
    _cr = saliency_correlation(_sal, _real_ref)
    _cosim_col.append(_cs)
    _iou_col.append(_tk["iou"])
    _pearson_col.append(_cr["pearson"])
    _spearman_col.append(_cr["spearman"])
    # Mismatch flag only applies to synthetic rows
    if row["is_synthetic"]:
        _mm = flag_confidence_evidence_mismatch(
            arcface_sim     = row["arcface_similarity"],
            sal_conc        = row["sal_concentration"],
            cos_sim_vs_real = _cs,
            bg_mass         = row.get("reg_background_v2", None),
            high_conf_thresh= _HIGH_CONF_THRESH,
        )
    else:
        _mm = False
    _mismatch_col.append(_mm)

forensic_df["cosim_vs_real"]               = _cosim_col
forensic_df["topk_iou_vs_real"]            = _iou_col
forensic_df["pearson_vs_real"]             = _pearson_col
forensic_df["spearman_vs_real"]            = _spearman_col
forensic_df["confidence_evidence_mismatch"]= _mismatch_col

_n_mm = forensic_df["confidence_evidence_mismatch"].sum()
print(f"\nPer-probe metrics added to forensic_df.")
print(f"Confidence-evidence mismatches: {_n_mm} / {forensic_df['is_synthetic'].sum()} synthetic rows")

In [ ]:
# =============================================================================
# Cell 30 — build_identity_forensic_summary() function
# =============================================================================

def build_identity_forensic_summary(identity, fdf, real_reference_mode="mean"):
    """Compute per-identity forensic summary.

    Parameters
    ----------
    identity             : str
    fdf                  : forensic_df
    real_reference_mode  : 'mean' (use real_sal_by_id mean map)

    Returns
    -------
    (summary_dict, real_df, synth_df)

    Notes
    -----
    real_sum contains exactly 1 real probe per identity, so within-identity
    real-real pairwise stats are not possible. Population real-real context
    comes from the intra_sims array computed in Cell 7.
    """
    real_df  = fdf[(fdf["identity"] == identity) & (~fdf["is_synthetic"])].copy()
    synth_df = fdf[(fdf["identity"] == identity) &  (fdf["is_synthetic"])].copy()

    s = {}  # summary dict
    s["identity"]  = identity
    s["n_real"]    = len(real_df)
    s["n_synth"]   = len(synth_df)

    # --- Real probe stats (single probe; population baseline from intra_sims) ---
    s["pop_real_real_cosim_mean"] = float(np.mean(intra_sims)) if len(intra_sims) else None
    s["pop_real_real_cosim_std"]  = float(np.std(intra_sims))  if len(intra_sims) else None

    if len(real_df) > 0:
        s["real_conc_mean"]    = float(real_df["sal_concentration"].mean())
        s["real_entropy_mean"] = float(real_df["sal_entropy"].mean())
    else:
        s["real_conc_mean"]    = None
        s["real_entropy_mean"] = None

    # --- Real-generated stats ---
    _sg = synth_df.dropna(subset=["cosim_vs_real"])
    if len(_sg) > 0:
        s["real_gen_cosim_mean"]   = float(_sg["cosim_vs_real"].mean())
        s["real_gen_cosim_std"]    = float(_sg["cosim_vs_real"].std())
        s["real_gen_iou_mean"]     = float(_sg["topk_iou_vs_real"].mean())
        s["real_gen_pearson_mean"] = float(_sg["pearson_vs_real"].mean())
        s["real_gen_spearman_mean"]= float(_sg["spearman_vs_real"].mean())
        s["synth_conc_mean"]       = float(synth_df["sal_concentration"].mean())
        s["synth_entropy_mean"]    = float(synth_df["sal_entropy"].mean())
    else:
        for k in ["real_gen_cosim_mean", "real_gen_cosim_std", "real_gen_iou_mean",
                  "real_gen_pearson_mean", "real_gen_spearman_mean",
                  "synth_conc_mean", "synth_entropy_mean"]:
            s[k] = None

    # --- Mismatch count ---
    s["n_mismatch"]   = int(synth_df["confidence_evidence_mismatch"].sum())
    s["pct_mismatch"] = (
        s["n_mismatch"] / len(synth_df) * 100 if len(synth_df) > 0 else 0.0
    )

    # --- Region means per group (7 regions) ---
    for r in REGION_NAMES_V2:
        col = f"reg_{r}_v2"
        s[f"region_real_{r}"]  = float(real_df[col].mean())  if len(real_df)  > 0 else None
        s[f"region_synth_{r}"] = float(synth_df[col].mean()) if len(synth_df) > 0 else None

    return s, real_df, synth_df


print("Cell 30: build_identity_forensic_summary() defined.")

In [ ]:
# =============================================================================
# Cell 31 — Run forensic summary for DEMO_IDENTITY
# =============================================================================

forensic_summary, _real_sub, _synth_sub = build_identity_forensic_summary(
    DEMO_IDENTITY, forensic_df)

# Print formatted summary
print(f"\n=== Forensic Summary: {DEMO_IDENTITY} ===")
print(f"  Real probes       : {forensic_summary['n_real']}")
print(f"  Synthetic probes  : {forensic_summary['n_synth']}")
print()
print("  -- Real probe saliency --")
print(f"     Concentration   : {forensic_summary['real_conc_mean']:.4f}" if forensic_summary['real_conc_mean'] is not None else "     Concentration   : N/A")
print(f"     Entropy         : {forensic_summary['real_entropy_mean']:.4f}" if forensic_summary['real_entropy_mean'] is not None else "     Entropy         : N/A")
print()
print("  -- Population real-real cosine sim (intra_sims baseline) --")
print(f"     Mean ± Std      : {forensic_summary['pop_real_real_cosim_mean']:.4f} ± {forensic_summary['pop_real_real_cosim_std']:.4f}")
print()
print("  -- Real-generated consistency --")
print(f"     Cosine sim mean : {forensic_summary['real_gen_cosim_mean']:.4f}" if forensic_summary['real_gen_cosim_mean'] is not None else "     Cosine sim mean : N/A")
print(f"     Top-10% IoU     : {forensic_summary['real_gen_iou_mean']:.4f}" if forensic_summary['real_gen_iou_mean'] is not None else "     Top-10% IoU     : N/A")
print(f"     Pearson r       : {forensic_summary['real_gen_pearson_mean']:.4f}" if forensic_summary['real_gen_pearson_mean'] is not None else "     Pearson r       : N/A")
print(f"     Spearman r      : {forensic_summary['real_gen_spearman_mean']:.4f}" if forensic_summary['real_gen_spearman_mean'] is not None else "     Spearman r      : N/A")
print()
print("  -- Synthetic saliency --")
print(f"     Concentration   : {forensic_summary['synth_conc_mean']:.4f}" if forensic_summary['synth_conc_mean'] is not None else "     Concentration   : N/A")
print(f"     Entropy         : {forensic_summary['synth_entropy_mean']:.4f}" if forensic_summary['synth_entropy_mean'] is not None else "     Entropy         : N/A")
print()
print(f"  -- Mismatch (high conf + suspicious saliency) --")
print(f"     Count           : {forensic_summary['n_mismatch']} / {forensic_summary['n_synth']} ({forensic_summary['pct_mismatch']:.1f}%)")
print()
print("  -- Region means (real | synth) --")
for r in REGION_NAMES_V2:
    rv = forensic_summary.get(f'region_real_{r}')
    sv = forensic_summary.get(f'region_synth_{r}')
    rv_s = f"{rv:.4f}" if rv is not None else "  N/A "
    sv_s = f"{sv:.4f}" if sv is not None else "  N/A "
    print(f"     {r:<12}: {rv_s} | {sv_s}")

# Save CSV
_summ_csv = os.path.join(FIG_DIR, f"forensic_identity_summary_{DEMO_IDENTITY}.csv")
pd.DataFrame([forensic_summary]).to_csv(_summ_csv, index=False)
print(f"\nSaved: {_summ_csv}")

In [ ]:
# =============================================================================
# Cell 32 — Forensic panel figure: saliency overlays for DEMO_IDENTITY
# =============================================================================
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

_id = DEMO_IDENTITY
_real_rows  = forensic_df[(forensic_df["identity"] == _id) & (~forensic_df["is_synthetic"])]
_synth_rows = forensic_df[(forensic_df["identity"] == _id) &  (forensic_df["is_synthetic"])].copy()

# Select synthetic examples: most similar + most divergent + mismatch if any
_synth_sorted = _synth_rows.sort_values("cosim_vs_real", ascending=False)
_most_sim   = _synth_sorted.iloc[:1]        # highest cosim
_most_div   = _synth_sorted.iloc[-1:]       # lowest cosim
_mismatch_rows = _synth_rows[_synth_rows["confidence_evidence_mismatch"]]

# Build ordered panel list: real probes, then most-similar synth, most-divergent synth,
# then one mismatch example if available
_panel_rows = []
for _, r in _real_rows.iterrows():
    _panel_rows.append((r, "real"))
for _, r in _most_sim.iterrows():
    _panel_rows.append((r, "most_similar"))
for _, r in _most_div.iterrows():
    _panel_rows.append((r, "most_divergent"))
if len(_mismatch_rows) > 0:
    _panel_rows.append((_mismatch_rows.iloc[0], "mismatch"))

n_cols = min(len(_panel_rows), 6)   # cap at 6 panels
_panel_rows = _panel_rows[:n_cols]

_type_color = {"real": "#2ecc71", "most_similar": "#3498db",
               "most_divergent": "#e74c3c", "mismatch": "#e67e22"}

def _auto_annotation(row, ptype):
    """Short forensic annotation derived from saliency stats."""
    if ptype == "real":
        top_r = max(REGION_NAMES_V2[:5],  # exclude background
                    key=lambda r: row.get(f"reg_{r}_v2", 0))
        return f"stable {top_r} focus"
    conc = row.get("sal_concentration", None)
    bg   = row.get("reg_background_v2", 0)
    cs   = row.get("cosim_vs_real", None)
    if ptype == "mismatch":
        return "high conf / suspicious evidence"
    if bg is not None and bg > _MISMATCH_BG_THRESH:
        return "drift toward background"
    if conc is not None and conc < _MISMATCH_CONC_THRESH:
        return "diffuse evidence"
    top_r = max(REGION_NAMES_V2[:5],
                key=lambda r: row.get(f"reg_{r}_v2", 0))
    if ptype == "most_similar":
        return f"preserved {top_r} emphasis"
    return f"drift toward {top_r}"

fig, axes = plt.subplots(1, n_cols, figsize=(3.5 * n_cols, 5))
if n_cols == 1:
    axes = [axes]

for ax, (row, ptype) in zip(axes, _panel_rows):
    sal = row["saliency_map"]
    ax.imshow(sal, cmap="inferno", vmin=0, vmax=sal.max() + 1e-9)
    ax.axis("off")

    # Subtitle
    _lbl = "REAL" if ptype == "real" else ptype.replace("_", " ").upper()
    _rank = row.get("rank1_match", None)
    _arc  = row.get("arcface_similarity", None)
    _cs   = row.get("cosim_vs_real", None)
    _lines = [f"{_lbl}  ({row['generation_method']})",
              f"rank1={'Y' if _rank else 'N'}" +
              (f"  arc={_arc:.3f}" if _arc is not None and not pd.isna(_arc) else "") +
              (f"  cos={_cs:.3f}" if _cs is not None and not pd.isna(_cs) else ""),
              _auto_annotation(row, ptype)]
    ax.set_title("\n".join(_lines), fontsize=7.5,
                  color=_type_color.get(ptype, "black"), pad=4)

plt.suptitle(f"Forensic panel — {_id}\n"
             f"(saliency maps; green=real, blue=most similar, red=most divergent, orange=mismatch)",
             fontsize=10, y=1.02)
plt.tight_layout()
_fig32_path = os.path.join(FIG_DIR, f"fig_forensic_panel_{_id}.png")
plt.savefig(_fig32_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {_fig32_path}")

In [ ]:
# =============================================================================
# Cell 33 — Region profile bar chart: real vs synthetic for DEMO_IDENTITY
# =============================================================================

_id = DEMO_IDENTITY
_real_sub_33  = forensic_df[(forensic_df["identity"] == _id) & (~forensic_df["is_synthetic"])]
_synth_sub_33 = forensic_df[(forensic_df["identity"] == _id) &  (forensic_df["is_synthetic"])]

_reg_cols_v2 = [f"reg_{r}_v2" for r in REGION_NAMES_V2]

_real_means  = _real_sub_33[_reg_cols_v2].mean() if len(_real_sub_33)  > 0 \
               else pd.Series({c: 0 for c in _reg_cols_v2})
_synth_means = _synth_sub_33[_reg_cols_v2].mean() if len(_synth_sub_33) > 0 \
               else pd.Series({c: 0 for c in _reg_cols_v2})

x = np.arange(len(REGION_NAMES_V2))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4.5))
bars_real  = ax.bar(x - width/2, _real_means.values,  width, label="Real probes",
                    color="#3498db", alpha=0.85, edgecolor="white")
bars_synth = ax.bar(x + width/2, _synth_means.values, width, label="Synthetic probes",
                    color="#e74c3c", alpha=0.85, edgecolor="white")

# Annotate bars that differ most (absolute delta > 0.04)
for xi, (rv, sv) in enumerate(zip(_real_means.values, _synth_means.values)):
    if abs(rv - sv) > 0.04:
        ax.annotate("*", xy=(xi + width/2, sv), ha="center", va="bottom",
                    fontsize=14, color="#c0392b")

ax.set_xticks(x)
ax.set_xticklabels(REGION_NAMES_V2, fontsize=10)
ax.set_ylabel("Mean fractional saliency mass", fontsize=10)
ax.set_title(f"Region saliency profile: {_id}\n"
             f"(* = real/synth difference > 0.04)", fontsize=11)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

_fig33_path = os.path.join(FIG_DIR, f"fig_region_profile_{_id}.png")
plt.savefig(_fig33_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {_fig33_path}")

In [ ]:
# =============================================================================
# Cell 34 — Real-vs-generated consistency table for DEMO_IDENTITY
# =============================================================================

_id = DEMO_IDENTITY
_synth_sub_34 = forensic_df[
    (forensic_df["identity"] == _id) & (forensic_df["is_synthetic"])
].copy()

# Per-probe table
_consist_rows = []
for _, row in _synth_sub_34.iterrows():
    _img = str(row["image_path"]).split("/")[-1].split("\\")[-1][:40]
    _consist_rows.append({
        "probe":          _img,
        "method":         row["generation_method"],
        "alpha":          row["blend_alpha"],
        "cosine_vs_real": row["cosim_vs_real"],
        "topk_iou":       row["topk_iou_vs_real"],
        "pearson_r":      row["pearson_vs_real"],
        "spearman_r":     row["spearman_vs_real"],
        "case":           row["case_label"],
        "arcface_sim":    row["arcface_similarity"],
    })

_consist_df = pd.DataFrame(_consist_rows)

# Save per-probe CSV
_consist_csv = os.path.join(FIG_DIR, f"forensic_consistency_{_id}.csv")
_consist_df.to_csv(_consist_csv, index=False)

# Aggregate 2-row comparison table
_pop_cosim_mean = float(np.mean(intra_sims)) if len(intra_sims) else float("nan")
_pop_cosim_std  = float(np.std(intra_sims))  if len(intra_sims) else float("nan")
_gen_cosim_mean = _consist_df["cosine_vs_real"].mean()
_gen_iou_mean   = _consist_df["topk_iou"].mean()
_gen_pear_mean  = _consist_df["pearson_r"].mean()
_gen_spear_mean = _consist_df["spearman_r"].mean()

print(f"=== Consistency report for {_id} ===")
print(f"{'Metric':<30} {'Pop real-real':>15} {'Real-generated':>15}")
print("-" * 62)
print(f"{'Cosine sim (mean)':<30} {_pop_cosim_mean:>15.4f} {_gen_cosim_mean:>15.4f}")
print(f"{'Cosine sim (std)':<30} {_pop_cosim_std:>15.4f} {_consist_df['cosine_vs_real'].std():>15.4f}")
print(f"{'Top-10% IoU (mean)':<30} {'(N/A)':>15} {_gen_iou_mean:>15.4f}")
print(f"{'Pearson r (mean)':<30} {'(N/A)':>15} {_gen_pear_mean:>15.4f}")
print(f"{'Spearman r (mean)':<30} {'(N/A)':>15} {_gen_spear_mean:>15.4f}")
print()
print(f"Note: 'Pop real-real' = population intra-identity baseline from {len(intra_sims)} pairs (Cell 7).")
print(f"      Per-probe detail saved → {_consist_csv}")

print(f"\nPer-probe preview (top 10):")
print(_consist_df[["probe","method","alpha","cosine_vs_real","topk_iou","pearson_r","case"]]
      .head(10).to_string(index=False))

In [ ]:
# =============================================================================
# Cell 35 — Confidence-evidence mismatch analysis for DEMO_IDENTITY
# =============================================================================

_id = DEMO_IDENTITY
_synth_sub_35 = forensic_df[
    (forensic_df["identity"] == _id) & (forensic_df["is_synthetic"])
].copy()

_mismatch_sub = _synth_sub_35[_synth_sub_35["confidence_evidence_mismatch"]].copy()
print(f"Mismatch rows for {_id}: {len(_mismatch_sub)} / {len(_synth_sub_35)}")

# Save mismatch CSV
_mm_csv = os.path.join(FIG_DIR, f"forensic_mismatch_{_id}.csv")
_mismatch_sub_csv = _mismatch_sub.drop(columns=["saliency_map"], errors="ignore")
_mismatch_sub_csv.to_csv(_mm_csv, index=False)
print(f"Saved mismatch CSV: {_mm_csv}")

# ── Scatter: ArcFace sim vs cosine-vs-real, with mismatch highlighted ──────
fig, ax = plt.subplots(figsize=(7, 5))

_all_syn = _synth_sub_35.dropna(subset=["arcface_similarity", "cosim_vs_real"])
_ok  = _all_syn[~_all_syn["confidence_evidence_mismatch"]]
_bad = _all_syn[ _all_syn["confidence_evidence_mismatch"]]

ax.scatter(_ok["arcface_similarity"],  _ok["cosim_vs_real"],
           s=40, alpha=0.6, color="steelblue", label="No mismatch", zorder=3)
ax.scatter(_bad["arcface_similarity"], _bad["cosim_vs_real"],
           s=70, alpha=0.9, color="#e74c3c", marker="^", label="Mismatch", zorder=4)

ax.axvline(_HIGH_CONF_THRESH, color="darkorange", ls="--", lw=1.5,
           label=f"High-conf thresh = {_HIGH_CONF_THRESH:.3f} (75th pct)")
ax.axhline(SIM_THRESHOLD, color="purple", ls=":", lw=1.5,
           label=f"Saliency sim thresh = {SIM_THRESHOLD:.3f}")

ax.set_xlabel("ArcFace similarity to true identity", fontsize=10)
ax.set_ylabel("Cosine sim: synthetic vs real saliency", fontsize=10)
ax.set_title(f"Confidence-evidence mismatch — {_id}\n"
             "(high confidence + suspicious saliency = mismatch)", fontsize=10)
ax.legend(fontsize=8); ax.grid(alpha=0.25)
plt.tight_layout()

_fig35_path = os.path.join(FIG_DIR, f"fig_confidence_mismatch_{_id}.png")
plt.savefig(_fig35_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {_fig35_path}")

# ── Optional mismatch saliency panel ────────────────────────────────────────
if len(_mismatch_sub) > 0:
    _n_show = min(len(_mismatch_sub), 4)
    fig2, axes2 = plt.subplots(1, _n_show, figsize=(3.5 * _n_show, 4))
    if _n_show == 1:
        axes2 = [axes2]
    for ax2, (_, mr) in zip(axes2, _mismatch_sub.head(_n_show).iterrows()):
        sal = mr["saliency_map"]
        ax2.imshow(sal, cmap="inferno", vmin=0, vmax=sal.max() + 1e-9)
        ax2.axis("off")
        _arc = mr.get("arcface_similarity", None)
        _cs  = mr.get("cosim_vs_real", None)
        ax2.set_title(
            f"{mr['generation_method']}\n"
            f"arc={_arc:.3f}" + (f"  cos={_cs:.3f}" if _cs is not None else ""),
            fontsize=8, color="#c0392b")
    plt.suptitle(f"Mismatch examples — {_id}", fontsize=10)
    plt.tight_layout()
    _fig35b_path = os.path.join(FIG_DIR, f"fig_confidence_mismatch_examples_{_id}.png")
    plt.savefig(_fig35b_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved mismatch examples: {_fig35b_path}")
else:
    print("No mismatch examples to display for this identity.")

In [ ]:
# =============================================================================
# Cell 36 — Auto-generate forensic text takeaways for DEMO_IDENTITY
# =============================================================================

_id = DEMO_IDENTITY
_s  = forensic_summary  # built in Cell 31

def _fmt(v, decimals=3):
    if v is None or (isinstance(v, float) and (v != v)):
        return "N/A"
    return f"{float(v):.{decimals}f}"

# Top-2 real regions
_real_reg_vals = {r: _s.get(f"region_real_{r}", 0) or 0 for r in REGION_NAMES_V2[:5]}
_top2_real = sorted(_real_reg_vals, key=_real_reg_vals.get, reverse=True)[:2]

# Top-2 synth regions
_synth_reg_vals = {r: _s.get(f"region_synth_{r}", 0) or 0 for r in REGION_NAMES_V2}
_top2_synth = sorted(_synth_reg_vals, key=_synth_reg_vals.get, reverse=True)[:2]

# Drift direction
_bg_synth  = _s.get("region_synth_background", 0) or 0
_bg_real   = _s.get("region_real_background", 0)  or 0
_bg_drift  = _bg_synth - _bg_real

# Case distribution for this identity
_synth_sub_36 = forensic_df[(forensic_df["identity"] == _id) & (forensic_df["is_synthetic"])]
_case_counts  = _synth_sub_36["case_label"].value_counts().to_dict()
_case_str = ", ".join(f"Case {c}: {_case_counts.get(c, 0)}" for c in ["A","B","C","D"])

bullets = [
    f"Identity: {_id}",
    f"Real probes concentrate saliency on {' and '.join(_top2_real)}, "
    f"with concentration={_fmt(_s.get('real_conc_mean'))} and entropy={_fmt(_s.get('real_entropy_mean'))}.",

    f"Population real-real cosine similarity baseline: "
    f"{_fmt(_s.get('pop_real_real_cosim_mean'))} ± {_fmt(_s.get('pop_real_real_cosim_std'))} "
    f"(from {len(intra_sims)} intra-identity pairs across all identities).",

    f"Real-generated saliency similarity: mean cosine = {_fmt(_s.get('real_gen_cosim_mean'))}, "
    f"top-10% IoU = {_fmt(_s.get('real_gen_iou_mean'))}, "
    f"Pearson r = {_fmt(_s.get('real_gen_pearson_mean'))}. "
    + ("Generated probes show notable saliency drift from real references."
       if (_s.get('real_gen_cosim_mean') or 1) < float(np.mean(intra_sims)) - float(np.std(intra_sims))
       else "Generated probes preserve saliency structure reasonably well."),

    (f"Synthetic probes shift {_bg_drift:+.4f} saliency mass toward background regions "
     "compared to real probes — suggesting artifact regions gain attention."
     if _bg_drift > 0.02
     else f"Synthetic probes do not show meaningful background saliency drift ({_bg_drift:+.4f})."),

    f"Synthetic saliency concentration = {_fmt(_s.get('synth_conc_mean'))}, "
    f"entropy = {_fmt(_s.get('synth_entropy_mean'))} "
    + ("(more diffuse than real probes — evidence is spread across more pixels)."
       if (_s.get('synth_entropy_mean') or 0) > (_s.get('real_entropy_mean') or 0)
       else "(similar concentration to real probes)."),

    f"Confidence-evidence mismatches: {_s['n_mismatch']} of {_s['n_synth']} synthetic probes "
    f"({_s['pct_mismatch']:.1f}%) show high ArcFace similarity but suspicious saliency.",

    f"Forensic case distribution: {_case_str}.",
]

# Print and save
print(f"\n=== Forensic Takeaways: {_id} ===")
for b in bullets:
    print(f"  • {b}")

_txt_path = os.path.join(FIG_DIR, f"forensic_takeaways_{_id}.txt")
with open(_txt_path, "w", encoding="utf-8") as _f:
    _f.write(f"Forensic Takeaways: {_id}\n")
    _f.write("=" * 60 + "\n\n")
    for b in bullets:
        _f.write(f"  • {b}\n")
print(f"\nSaved: {_txt_path}")

In [ ]:
# =============================================================================
# Cell 37 — Multi-identity drift ranking (all overlap identities)
# =============================================================================

_drift_rows = []
_overlap_ids = [
    _id for _id in forensic_df["identity"].unique()
    if _id in real_sal_by_id
    and forensic_df[(forensic_df["identity"]==_id) & forensic_df["is_synthetic"]].shape[0] >= 1
]

print(f"Running drift summary for {len(_overlap_ids)} identities...")

for _id in _overlap_ids:
    _s37, _, _sd37 = build_identity_forensic_summary(_id, forensic_df)
    _drift_rows.append({
        "identity":          _id,
        "n_synth":           _s37["n_synth"],
        "real_gen_cosim":    _s37.get("real_gen_cosim_mean"),
        "real_gen_iou":      _s37.get("real_gen_iou_mean"),
        "synth_conc":        _s37.get("synth_conc_mean"),
        "n_mismatch":        _s37["n_mismatch"],
        "pct_mismatch":      _s37["pct_mismatch"],
        "region_bg_synth":   _s37.get("region_synth_background"),
    })

drift_ranking_df = pd.DataFrame(_drift_rows).dropna(subset=["real_gen_cosim"])
drift_ranking_df = drift_ranking_df.sort_values("real_gen_cosim")  # ascending = most drifted
drift_ranking_df.to_csv(os.path.join(FIG_DIR, "forensic_drift_ranking_all.csv"), index=False)

print(f"Drift ranking computed for {len(drift_ranking_df)} identities.")
print(f"Most drifted identity : {drift_ranking_df.iloc[0]['identity']}  "
      f"(cosim={drift_ranking_df.iloc[0]['real_gen_cosim']:.4f})")
print(f"Least drifted identity: {drift_ranking_df.iloc[-1]['identity']}  "
      f"(cosim={drift_ranking_df.iloc[-1]['real_gen_cosim']:.4f})")

# Horizontal bar chart — top 20 most drifted identities
_top20 = drift_ranking_df.head(20).copy()
fig, ax = plt.subplots(figsize=(9, max(4, len(_top20) * 0.4)))
_colors = ["#e74c3c" if row["identity"] == DEMO_IDENTITY else "steelblue"
           for _, row in _top20.iterrows()]
ax.barh(_top20["identity"], _top20["real_gen_cosim"], color=_colors, edgecolor="white")
ax.axvline(float(np.mean(intra_sims)), color="green", ls="--", lw=1.5,
           label=f"Pop real-real mean = {np.mean(intra_sims):.3f}")
ax.set_xlabel("Mean cosine sim (real vs generated saliency)", fontsize=10)
ax.set_title("Top-20 most drifted identities\n"
             "(lower = synthetic saliency diverges most from real reference)", fontsize=10)
ax.legend(fontsize=9); ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
_fig37_path = os.path.join(FIG_DIR, "fig_drift_ranking_top20.png")
plt.savefig(_fig37_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {_fig37_path}")

In [ ]:
# =============================================================================
# Cell 38 — Output file validation checklist
# =============================================================================

_id = DEMO_IDENTITY
_expected = [
    ("forensic_identity_summary",   f"forensic_identity_summary_{_id}.csv"),
    ("forensic panel figure",        f"fig_forensic_panel_{_id}.png"),
    ("region profile figure",        f"fig_region_profile_{_id}.png"),
    ("consistency CSV",             f"forensic_consistency_{_id}.csv"),
    ("mismatch CSV",                f"forensic_mismatch_{_id}.csv"),
    ("mismatch scatter figure",      f"fig_confidence_mismatch_{_id}.png"),
    ("text takeaways",              f"forensic_takeaways_{_id}.txt"),
    ("drift ranking CSV",           "forensic_drift_ranking_all.csv"),
    ("drift ranking figure",        "fig_drift_ranking_top20.png"),
]

print(f"=== Output file checklist for {_id} ===")
_all_ok = True
for label, fname in _expected:
    path = os.path.join(FIG_DIR, fname)
    exists = os.path.isfile(path)
    status = "OK" if exists else "MISSING"
    if not exists:
        _all_ok = False
    print(f"  [{status:7s}] {label:<35} {fname}")

print()
if _all_ok:
    print("All auditor output files present.")
else:
    print("WARNING: Some output files are missing. Re-run the relevant cells.")

# Diagnostic summary
print(f"\nDiagnostic summary:")
print(f"  forensic_df rows       : {len(forensic_df)}")
print(f"  Real rows              : {(~forensic_df['is_synthetic']).sum()}")
print(f"  Synthetic rows         : {forensic_df['is_synthetic'].sum()}")
print(f"  Identities covered     : {forensic_df['identity'].nunique()}")
print(f"  Mismatch rows (total)  : {forensic_df['confidence_evidence_mismatch'].sum()}")
print(f"  Drift ranking entries  : {len(drift_ranking_df)}")

---
## Auditor Section: Notes

### What was produced

For the selected demo identity, this section generated:

| File | Purpose |
|------|---------|
| `forensic_identity_summary_<id>.csv` | Aggregate saliency stats |
| `fig_forensic_panel_<id>.png` | Side-by-side real vs synthetic saliency overlays |
| `fig_region_profile_<id>.png` | 7-region comparison bar chart |
| `forensic_consistency_<id>.csv` | Per-probe cosine/IoU/correlation table |
| `forensic_mismatch_<id>.csv` | High-confidence suspicious-saliency rows |
| `fig_confidence_mismatch_<id>.png` | Mismatch scatter plot |
| `forensic_takeaways_<id>.txt` | Auto-generated 8-bullet summary |
| `forensic_drift_ranking_all.csv` | All-identity real-generated drift ranking |
| `fig_drift_ranking_top20.png` | Bar chart of top-20 most drifted identities |

### Data constraint note

`real_sum` contains **exactly 1 real probe per identity**, so within-identity real-real
pairwise statistics cannot be computed from this dataframe alone. The *population*
real-real baseline reported here uses `intra_sims` from Cell 7, which was derived
from multi-probe identities in `results/crise_maps/`.

### Method-agnostic design

All helper functions (Cells 27–30) accept raw saliency arrays directly and carry no
hardcoded CRISE references. To compare RISE baseline saliency, add a `rise_saliency_path`
column to `forensic_df` and repoint the single `saliency_col` variable in Cell 34.
